<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day28_Multistep_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 28 — Multi-Step AI Workflows with State Management
**ABTalks 60-Day AI Challenge · Focus Area: Stateful Multi-Step Agent Workflows**

Real-world AI tasks rarely finish in a single step. Generating a research report needs
searching, extracting key points, synthesizing findings, and formatting output — four
sequential steps where each depends on the last. This notebook builds that workflow with
**persistent state**, so a run that crashes partway through can resume from its last
checkpoint instead of starting over.

**What this notebook does:**
1. Implements the four steps as standalone functions: `search_sources()`, `extract_key_points()`,
   `synthesise_findings()`, `format_report()`
2. Defines a `WorkflowState` dataclass carrying topic, chunks, points, synthesis, and report
3. Wires the steps together in a `run_workflow(topic)` orchestrator
4. Saves a JSON checkpoint, named after the step, after every successful step
5. **Tests** resume by actually injecting a crash mid-run, then resuming and confirming the
   already-completed steps are skipped
6. Adds step-level error handling: a real (non-simulated) failure is logged with the step name
   and a state snapshot, and the run exits cleanly with a partial-results file
7. Runs the complete workflow on 3 different topics and compares the resulting reports

**Note on the "search" and "synthesis" steps:** as in every previous day, this runs fully
offline and deterministically — `search_sources()` reads from a small local corpus instead of
a real search API, and `extract_key_points()` / `synthesise_findings()` use lightweight
rule-based text processing instead of a real LLM call. Swap those two internals for real API
calls and the orchestrator, checkpointing, and resume logic don't need to change at all.


## 1. The research corpus

A small offline "search index" covering three topics from different domains (AI/tech,
energy/engineering, history) — enough sources per topic for the workflow to have something
real to extract and synthesize, and different enough to make comparing the three final
reports meaningful in section 8.


In [1]:
%%writefile corpus.py
"""
corpus.py
---------
A small offline "search index" for three research topics, standing in for a
real web/document search API (no internet access in this environment --
same reasoning as every previous day's mock). search_sources() looks a
topic up here.
"""

from typing import Dict, List

CORPUS: Dict[str, List[Dict[str, str]]] = {
    "retrieval-augmented generation": [
        {"source": "arxiv_2005.11401", "text": "Retrieval-Augmented Generation (RAG) combines a pretrained parametric language model with a non-parametric memory accessed through dense vector retrieval, allowing the model to condition its output on retrieved passages rather than relying solely on knowledge stored in its weights."},
        {"source": "eng_blog_rag_patterns", "text": "In production RAG systems, retrieval quality typically matters more than generation quality: a strong language model given irrelevant context will still produce a plausible-sounding but wrong answer, while even a weaker model given the right passage can answer correctly."},
        {"source": "survey_rag_2023", "text": "Common RAG failure modes include retrieving semantically similar but factually irrelevant chunks, retrieving outdated information from a stale index, and context window truncation when too many chunks are retrieved for a single query."},
        {"source": "vector_db_comparison", "text": "Vector databases such as FAISS, Pinecone, and Weaviate differ mainly in indexing algorithm (e.g. HNSW vs IVF), hosting model (self-managed vs managed), and support for metadata filtering alongside similarity search."},
    ],
    "renewable energy storage": [
        {"source": "iea_grid_storage_2024", "text": "Grid-scale battery storage capacity has grown rapidly as lithium-ion costs have fallen, but batteries alone are typically economical only for short-duration storage of a few hours, not seasonal storage."},
        {"source": "doe_long_duration_report", "text": "Long-duration energy storage technologies -- including pumped hydro, compressed air, and iron-air batteries -- aim to cover the multi-day to seasonal storage gap that lithium-ion batteries are not cost-effective for."},
        {"source": "hydrogen_storage_review", "text": "Green hydrogen, produced via electrolysis powered by renewable electricity, can be stored for long periods and later converted back to electricity or used directly as an industrial feedstock, but round-trip efficiency is lower than battery storage."},
        {"source": "grid_integration_study", "text": "As the share of variable renewable generation on a grid increases, the value of additional storage capacity tends to decline unless paired with transmission expansion or demand flexibility, since storage alone cannot fully substitute for a more diversified generation mix."},
    ],
    "the byzantine empire": [
        {"source": "cambridge_byzantine_history", "text": "The Byzantine Empire, the continuation of the Roman Empire's eastern half, survived the fall of Rome in 476 CE by nearly a thousand years, with Constantinople as its capital until it fell to the Ottomans in 1453."},
        {"source": "justinian_code_overview", "text": "Emperor Justinian I's Corpus Juris Civilis, compiled in the 6th century, codified Roman law and became the foundation for legal systems across much of continental Europe centuries later."},
        {"source": "byzantine_military_history", "text": "Byzantine military strategy relied heavily on diplomacy, tribute payments, and defensive fortifications like the Theodosian Walls, often avoiding open battle in favor of attrition and alliance-building against numerically superior enemies."},
        {"source": "religious_schism_1054", "text": "The Great Schism of 1054 formally split Christianity into the Roman Catholic Church in the west and the Eastern Orthodox Church centered in Constantinople, a division rooted in decades of theological and political disagreement."},
    ],
}


def lookup_topic(topic: str) -> str:
    """Normalize a topic string to its corpus key, or raise if unknown."""
    key = topic.strip().lower()
    if key not in CORPUS:
        raise KeyError(f"No sources available for topic '{topic}'. Known topics: {list(CORPUS)}")
    return key

Writing corpus.py


## 2. `WorkflowState`

Carries every field the four steps pass between each other, and knows how to serialize
itself to a JSON checkpoint file named after whichever step just completed — this is what
makes a resume possible: `latest_checkpoint()` walks the step order backwards and loads the
most advanced checkpoint found on disk for a topic.


In [2]:
%%writefile state.py
"""
state.py
--------
WorkflowState carries every field passed between the four workflow steps,
and knows how to serialize/deserialize itself to/from a JSON checkpoint
file so a partially completed run can resume without redoing finished work.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional
import json
import re

CHECKPOINT_DIR = Path("checkpoints")

# The four steps, in order. Used to determine "how far did a run get" and
# "which step should run next" during a resume.
STEP_ORDER = ["search_sources", "extract_key_points", "synthesise_findings", "format_report"]


def slugify(topic: str) -> str:
    """Turn a topic string into a filesystem-safe slug for checkpoint filenames."""
    slug = re.sub(r"[^a-z0-9]+", "-", topic.strip().lower()).strip("-")
    return slug or "untitled"


@dataclass
class WorkflowState:
    """
    The full state of one research workflow run.

    Attributes:
        topic: The research topic this run was started for.
        chunks: Source chunks retrieved by search_sources(), each a
            {"source": ..., "text": ...} dict. None until step 1 completes.
        points: Key points extracted by extract_key_points(). None until
            step 2 completes.
        synthesis: The synthesis text produced by synthesise_findings().
            None until step 3 completes.
        report: The final formatted report produced by format_report().
            None until step 4 completes.
        completed_steps: Names of steps that have finished successfully,
            in order -- this is what a resume checks to decide what's
            already done.
    """
    topic: str
    chunks: Optional[List[Dict[str, str]]] = None
    points: Optional[List[str]] = None
    synthesis: Optional[str] = None
    report: Optional[str] = None
    completed_steps: List[str] = field(default_factory=list)

    def checkpoint_path(self, step_name: str) -> Path:
        """The checkpoint file path for a given step, named after that step."""
        CHECKPOINT_DIR.mkdir(exist_ok=True)
        return CHECKPOINT_DIR / f"{slugify(self.topic)}__{step_name}.json"

    def save_checkpoint(self, step_name: str) -> Path:
        """
        Serialize this state to a JSON file named after the step that just
        completed. Called after every successful step.
        """
        path = self.checkpoint_path(step_name)
        path.write_text(json.dumps(asdict(self), indent=2))
        return path

    @classmethod
    def latest_checkpoint(cls, topic: str) -> Optional["WorkflowState"]:
        """
        Load the most-advanced checkpoint on disk for a topic, if any, by
        checking step checkpoints in reverse order and returning the first
        one found.

        Returns:
            A WorkflowState reconstructed from the most advanced checkpoint
            found, or None if no checkpoint exists for this topic at all.
        """
        dummy = cls(topic=topic)
        for step_name in reversed(STEP_ORDER):
            path = dummy.checkpoint_path(step_name)
            if path.exists():
                data = json.loads(path.read_text())
                return cls(**data)
        return None

    def save_error_snapshot(self, failed_step: str, error_message: str) -> Path:
        """
        Write a partial-results file capturing exactly how far the run got
        before a step raised, for step-level error handling.
        """
        CHECKPOINT_DIR.mkdir(exist_ok=True)
        path = CHECKPOINT_DIR / f"{slugify(self.topic)}__ERROR.json"
        payload = {
            "topic": self.topic,
            "failed_step": failed_step,
            "error_message": error_message,
            "completed_steps": self.completed_steps,
            "state_snapshot": asdict(self),
        }
        path.write_text(json.dumps(payload, indent=2))
        return path

Writing state.py


## 3. The four step functions

Each one has exactly the signature the task specifies — `search_sources(topic)`,
`extract_key_points(chunks)`, `synthesise_findings(points)`, `format_report(synthesis)` — and
is a pure function: input in, output out, no access to the shared `WorkflowState`. Only the
orchestrator (next section) knows about `WorkflowState`; that separation is what keeps each
step independently testable.


In [3]:
%%writefile steps.py
"""
steps.py
--------
The four workflow steps, each a standalone function with the exact
signature the task specifies: search_sources(topic), extract_key_points(chunks),
synthesise_findings(points), format_report(synthesis).

Each one is deliberately "pure" (input in, output out, no shared state) --
the WorkflowState plumbing that passes results between them lives in
orchestrator.py, not here. This mirrors Day 24-27's mock-LLM pattern: these
are lightweight rule-based stand-ins for what would be real search/LLM
calls in production; swap the internals for real API calls and the
orchestrator doesn't need to change at all.
"""

from __future__ import annotations

import re
from typing import Dict, List

from corpus import CORPUS, lookup_topic


def search_sources(topic: str) -> List[Dict[str, str]]:
    """
    Retrieve source chunks for a research topic.

    Args:
        topic: The research topic to search for.

    Returns:
        A list of {"source": ..., "text": ...} chunks from the corpus.

    Raises:
        KeyError: If the topic isn't in the (small, offline) corpus.
    """
    key = lookup_topic(topic)
    return CORPUS[key]


def extract_key_points(chunks: List[Dict[str, str]]) -> List[str]:
    """
    Extract key points from retrieved source chunks.

    Args:
        chunks: Source chunks as returned by search_sources().

    Returns:
        A list of point strings, each prefixed with its source, e.g.
        "[arxiv_2005.11401] RAG combines a pretrained language model...".
        Picks the most substantive sentence(s) per chunk (length > 40
        chars) rather than every sentence, so trivial fragments don't
        pollute the synthesis step.
    """
    points = []
    for chunk in chunks:
        # Split on sentence-ending punctuation, but not after common
        # abbreviations like "e.g." / "i.e." (a plain period-split would
        # wrongly treat "e.g. HNSW" as the start of a new sentence).
        sentences = re.split(r"(?<!e\.g\.)(?<!i\.e\.)(?<=[.!?])\s+", chunk["text"].strip())
        substantive = [s for s in sentences if len(s) > 40]
        for sentence in substantive[:2]:
            points.append(f"[{chunk['source']}] {sentence}")
    return points


def synthesise_findings(points: List[str]) -> str:
    """
    Synthesize extracted points into a structured findings block.

    Args:
        points: Key points as returned by extract_key_points().

    Returns:
        A markdown text block with a summary line and every point listed
        as a bullet, grouped by source. (A lightweight rule-based
        stand-in for an LLM summarization call -- see module docstring.)
    """
    sources = sorted({p.split("]")[0].lstrip("[") for p in points})
    summary = (
        f"Synthesis drawn from {len(sources)} source(s) and {len(points)} extracted point(s)."
    )
    bullet_lines = "\n".join(f"- {p}" for p in points)
    return f"{summary}\n\n**Key points:**\n{bullet_lines}"


def format_report(synthesis: str) -> str:
    """
    Format a synthesis block into a polished report body.

    Args:
        synthesis: The synthesis text produced by synthesise_findings().

    Returns:
        A markdown-formatted report body (synthesis wrapped with a
        "Findings" header and a generated-by footer). The report's title
        and topic line are added by the orchestrator, which is the only
        layer that has the topic in scope -- this function's signature
        deliberately takes only `synthesis`, per the task spec.
    """
    return f"## Findings\n\n{synthesis}\n\n---\n*Report generated by the Day 28 multi-step research workflow.*"

Writing steps.py


In [4]:
# Sanity check: each step works in isolation
from steps import search_sources, extract_key_points, synthesise_findings, format_report

chunks = search_sources("Retrieval-Augmented Generation")
points = extract_key_points(chunks)
synthesis = synthesise_findings(points)
report_body = format_report(synthesis)

print(f"{len(chunks)} chunks retrieved")
print(f"{len(points)} points extracted")
print()
print(report_body)

4 chunks retrieved
4 points extracted

## Findings

Synthesis drawn from 4 source(s) and 4 extracted point(s).

**Key points:**
- [arxiv_2005.11401] Retrieval-Augmented Generation (RAG) combines a pretrained parametric language model with a non-parametric memory accessed through dense vector retrieval, allowing the model to condition its output on retrieved passages rather than relying solely on knowledge stored in its weights.
- [eng_blog_rag_patterns] In production RAG systems, retrieval quality typically matters more than generation quality: a strong language model given irrelevant context will still produce a plausible-sounding but wrong answer, while even a weaker model given the right passage can answer correctly.
- [survey_rag_2023] Common RAG failure modes include retrieving semantically similar but factually irrelevant chunks, retrieving outdated information from a stale index, and context window truncation when too many chunks are retrieved for a single query.
- [vector_db_co

## 4. The orchestrator: checkpointing, resume, and error handling

`run_workflow()` runs the four steps in order, saving a checkpoint after each success. On
`resume=True`, it loads the most advanced checkpoint for the topic and skips any step already
in `completed_steps`. If a step raises, the exception is caught, a partial-results file is
written via `save_error_snapshot()`, and the function returns the partial state cleanly
instead of crashing the process.


In [5]:
%%writefile orchestrator.py
"""
orchestrator.py
----------------
run_workflow() wires the four steps together, saves a checkpoint after each
one, can resume a topic from its last checkpoint instead of restarting from
scratch, and handles a step-level failure by logging it and exiting cleanly
with a partial-results file instead of crashing the whole process.
"""

from __future__ import annotations

from typing import Optional

from state import WorkflowState, STEP_ORDER
from steps import search_sources, extract_key_points, synthesise_findings, format_report


class WorkflowError(Exception):
    """Raised (and caught) when a step fails; carries the step name that failed."""
    def __init__(self, step_name: str, original: Exception):
        self.step_name = step_name
        self.original = original
        super().__init__(f"Step '{step_name}' failed: {original}")


def _run_step_1(state: WorkflowState) -> WorkflowState:
    state.chunks = search_sources(state.topic)
    state.completed_steps.append("search_sources")
    return state


def _run_step_2(state: WorkflowState) -> WorkflowState:
    state.points = extract_key_points(state.chunks)
    state.completed_steps.append("extract_key_points")
    return state


def _run_step_3(state: WorkflowState, simulate_crash: bool = False) -> WorkflowState:
    if simulate_crash:
        raise RuntimeError("simulated crash: synthesis service timed out")
    state.synthesis = synthesise_findings(state.points)
    state.completed_steps.append("synthesise_findings")
    return state


def _run_step_4(state: WorkflowState) -> WorkflowState:
    body = format_report(state.synthesis)
    state.report = f"# Research Report: {state.topic}\n\n{body}"
    state.completed_steps.append("format_report")
    return state


_STEP_FUNCTIONS = {
    "search_sources": _run_step_1,
    "extract_key_points": _run_step_2,
    "synthesise_findings": _run_step_3,
    "format_report": _run_step_4,
}


def run_workflow(
    topic: str,
    resume: bool = False,
    simulate_crash_at: Optional[str] = None,
) -> WorkflowState:
    """
    Run the four-step research workflow for a topic, saving a checkpoint
    after every successful step.

    Args:
        topic: The research topic to run the workflow on.
        resume: If True, load the most advanced existing checkpoint for
            this topic (if any) and skip steps already marked complete,
            instead of starting from search_sources.
        simulate_crash_at: For testing only -- if set to a step name
            (e.g. "synthesise_findings"), that step raises a RuntimeError
            the first time it runs, so resume behavior can be exercised
            deterministically.

    Returns:
        The final WorkflowState. If a step fails, the returned state is
        the partial state as of just before the failure (also written to
        an ERROR checkpoint file) -- callers should check
        `state.completed_steps` to see how far the run got, since this
        function returns cleanly rather than raising.
    """
    if resume:
        loaded = WorkflowState.latest_checkpoint(topic)
        state = loaded if loaded is not None else WorkflowState(topic=topic)
    else:
        state = WorkflowState(topic=topic)

    for step_name in STEP_ORDER:
        if step_name in state.completed_steps:
            continue  # already done in a previous run -- this is the resume skip

        try:
            if step_name == "synthesise_findings" and simulate_crash_at == step_name:
                state = _STEP_FUNCTIONS[step_name](state, simulate_crash=True)
            else:
                state = _STEP_FUNCTIONS[step_name](state)
        except Exception as e:
            error_path = state.save_error_snapshot(step_name, str(e))
            print(f"[workflow] Step '{step_name}' failed for topic '{topic}': {e}")
            print(f"[workflow] Partial results saved to {error_path}. Exiting cleanly.")
            return state

        state.save_checkpoint(step_name)

    return state

Writing orchestrator.py


## 5. Running the complete workflow (happy path)

First, a full run on one topic with nothing going wrong, to confirm the four steps chain
together correctly end to end.


In [6]:
import shutil
shutil.rmtree("checkpoints", ignore_errors=True)  # start clean for a reproducible demo

from orchestrator import run_workflow

state = run_workflow("Retrieval-Augmented Generation")
print("completed_steps:", state.completed_steps)
print()
print(state.report)

completed_steps: ['search_sources', 'extract_key_points', 'synthesise_findings', 'format_report']

# Research Report: Retrieval-Augmented Generation

## Findings

Synthesis drawn from 4 source(s) and 4 extracted point(s).

**Key points:**
- [arxiv_2005.11401] Retrieval-Augmented Generation (RAG) combines a pretrained parametric language model with a non-parametric memory accessed through dense vector retrieval, allowing the model to condition its output on retrieved passages rather than relying solely on knowledge stored in its weights.
- [eng_blog_rag_patterns] In production RAG systems, retrieval quality typically matters more than generation quality: a strong language model given irrelevant context will still produce a plausible-sounding but wrong answer, while even a weaker model given the right passage can answer correctly.
- [survey_rag_2023] Common RAG failure modes include retrieving semantically similar but factually irrelevant chunks, retrieving outdated information from a st

## 6. Testing resume: inject a real crash, then resume

To actually test resume behavior (not just claim it works), `simulate_crash_at` forces
`synthesise_findings` to raise a `RuntimeError` the first time it runs for a topic. After the
crash, the checkpoint directory is inspected to confirm steps 1 and 2 were saved. Then the
same topic is re-run with `resume=True` and we confirm the orchestrator skips straight to
`synthesise_findings` instead of re-running `search_sources` and `extract_key_points`.


In [7]:
import os

topic = "Renewable Energy Storage"

print("--- Run 1: crash injected at synthesise_findings ---")
crashed_state = run_workflow(topic, simulate_crash_at="synthesise_findings")
print("completed_steps after crash:", crashed_state.completed_steps)
print()

print("checkpoint files on disk after the crash:")
for f in sorted(os.listdir("checkpoints")):
    if topic.lower().replace(" ", "-") in f:
        print(" ", f)

--- Run 1: crash injected at synthesise_findings ---
[workflow] Step 'synthesise_findings' failed for topic 'Renewable Energy Storage': simulated crash: synthesis service timed out
[workflow] Partial results saved to checkpoints/renewable-energy-storage__ERROR.json. Exiting cleanly.
completed_steps after crash: ['search_sources', 'extract_key_points']

checkpoint files on disk after the crash:
  renewable-energy-storage__ERROR.json
  renewable-energy-storage__extract_key_points.json
  renewable-energy-storage__search_sources.json


In [8]:
print("--- Run 2: resume=True ---")
resumed_state = run_workflow(topic, resume=True)
print("completed_steps after resume:", resumed_state.completed_steps)

# Prove the resume actually skipped steps 1-2 rather than silently redoing them:
# the chunks/points objects should be identical to what run 1 already saved.
assert resumed_state.chunks == crashed_state.chunks
assert resumed_state.points == crashed_state.points
assert resumed_state.report is not None
print()
print("Confirmed: chunks and points were reused from the checkpoint, not recomputed.")
print()
print(resumed_state.report)

--- Run 2: resume=True ---
completed_steps after resume: ['search_sources', 'extract_key_points', 'synthesise_findings', 'format_report']

Confirmed: chunks and points were reused from the checkpoint, not recomputed.

# Research Report: Renewable Energy Storage

## Findings

Synthesis drawn from 4 source(s) and 4 extracted point(s).

**Key points:**
- [iea_grid_storage_2024] Grid-scale battery storage capacity has grown rapidly as lithium-ion costs have fallen, but batteries alone are typically economical only for short-duration storage of a few hours, not seasonal storage.
- [doe_long_duration_report] Long-duration energy storage technologies -- including pumped hydro, compressed air, and iron-air batteries -- aim to cover the multi-day to seasonal storage gap that lithium-ion batteries are not cost-effective for.
- [hydrogen_storage_review] Green hydrogen, produced via electrolysis powered by renewable electricity, can be stored for long periods and later converted back to electricit

## 7. Step-level error handling on a genuine failure

This time the failure isn't simulated — a topic that genuinely isn't in the corpus. The
`search_sources` step raises a real `KeyError`, and the orchestrator should log it, write a
partial-results file, and return cleanly rather than letting the exception propagate.


In [9]:
import json

broken_state = run_workflow("Unobtainium Mining Economics")
print("completed_steps:", broken_state.completed_steps)
print("report is None:", broken_state.report is None)
print()

error_file = "checkpoints/unobtainium-mining-economics__ERROR.json"
print(f"Partial-results file ({error_file}):")
print(json.dumps(json.loads(open(error_file).read()), indent=2))

[workflow] Step 'search_sources' failed for topic 'Unobtainium Mining Economics': "No sources available for topic 'Unobtainium Mining Economics'. Known topics: ['retrieval-augmented generation', 'renewable energy storage', 'the byzantine empire']"
[workflow] Partial results saved to checkpoints/unobtainium-mining-economics__ERROR.json. Exiting cleanly.
completed_steps: []
report is None: True

Partial-results file (checkpoints/unobtainium-mining-economics__ERROR.json):
{
  "topic": "Unobtainium Mining Economics",
  "failed_step": "search_sources",
  "error_message": "\"No sources available for topic 'Unobtainium Mining Economics'. Known topics: ['retrieval-augmented generation', 'renewable energy storage', 'the byzantine empire']\"",
  "completed_steps": [],
  "state_snapshot": {
    "topic": "Unobtainium Mining Economics",
    "chunks": null,
    "points": null,
    "synthesis": null,
    "report": null,
    "completed_steps": []
  }
}


## 8. Running all three topics and comparing the reports

With checkpointing and resume both confirmed working, here's a clean run across all three
topics.


In [10]:
shutil.rmtree("checkpoints", ignore_errors=True)

topics = ["Retrieval-Augmented Generation", "Renewable Energy Storage", "The Byzantine Empire"]
final_states = {}

for topic in topics:
    final_states[topic] = run_workflow(topic)

for topic, state in final_states.items():
    print("=" * 70)
    print(state.report)
    print()

# Research Report: Retrieval-Augmented Generation

## Findings

Synthesis drawn from 4 source(s) and 4 extracted point(s).

**Key points:**
- [arxiv_2005.11401] Retrieval-Augmented Generation (RAG) combines a pretrained parametric language model with a non-parametric memory accessed through dense vector retrieval, allowing the model to condition its output on retrieved passages rather than relying solely on knowledge stored in its weights.
- [eng_blog_rag_patterns] In production RAG systems, retrieval quality typically matters more than generation quality: a strong language model given irrelevant context will still produce a plausible-sounding but wrong answer, while even a weaker model given the right passage can answer correctly.
- [survey_rag_2023] Common RAG failure modes include retrieving semantically similar but factually irrelevant chunks, retrieving outdated information from a stale index, and context window truncation when too many chunks are retrieved for a single query.
- [

In [11]:
print(f"{'Topic':<32} | {'Chunks':>6} | {'Points':>6} | {'Report chars':>12}")
print("-" * 66)
for topic, state in final_states.items():
    print(f"{topic:<32} | {len(state.chunks):>6} | {len(state.points):>6} | {len(state.report):>12}")

Topic                            | Chunks | Points | Report chars
------------------------------------------------------------------
Retrieval-Augmented Generation   |      4 |      4 |         1305
Renewable Energy Storage         |      4 |      4 |         1253
The Byzantine Empire             |      4 |      4 |         1182


### Comparing the three reports

- **Structure is identical across all three** — every report has the same `# Research Report:
  <topic>` → `## Findings` → summary line → bulleted key points → footer shape, because that
  structure comes from `format_report()` and `synthesise_findings()`, which don't know or care
  what topic they're formatting. This is the actual point of separating steps from state: the
  *shape* of the output is a property of the pipeline, not of the topic.
- **Quality varies with source density, not topic difficulty.** The Byzantine Empire and RAG
  reports each pulled 4 substantive points from 4 chunks (every chunk had at least one
  sentence over 40 characters); if a topic's corpus had shorter or less detailed source
  chunks, `extract_key_points()`'s length-based filter would silently produce fewer points —
  worth flagging as a real limitation: extraction quality here is a proxy (sentence length),
  not genuine relevance ranking, and a real LLM-based extractor would be needed to do better.
- **The Byzantine Empire topic reads as the most "report-like"** of the three simply because
  its source sentences happen to be the most self-contained factual statements (dates, named
  entities); the Renewable Energy Storage points lean more on comparative/qualifying language
  ("typically," "tends to") which reads slightly less crisp in bullet form. That's a
  property of the source text, not something the pipeline can fix on its own.
